# 06. 장기 메모리 & 스킬

Deep Agents에서 **장기 메모리**와 **스킬**을 활용하여 에이전트의 지식과 능력을 확장하는 방법을 학습합니다.
- 장기 메모리: 대화 간 정보 유지 (`CompositeBackend` + `StoreBackend`)
- 스킬: 도메인 전문 지식의 모듈화 (`SKILL.md` 기반 점진적 공개)

## 학습 목표
- `CompositeBackend` + `StoreBackend`로 장기 메모리를 구현한다
- 크로스 스레드 메모리 공유 패턴을 이해한다
- `AGENTS.md`를 통해 에이전트에 컨텍스트를 주입한다
- 스킬(SKILL.md)의 구조와 Progressive Disclosure를 이해한다
- Skills vs Memory의 차이를 파악한다

In [7]:
# 환경 설정: .env 파일에서 API 키를 로드합니다.
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

환경 설정 완료


In [8]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — 


In [9]:
# gpt-5.4-mini: 스킬 노트북에서 사용할 LLM (빠르고 비용 효율적)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")

---
## 1. 장기 메모리가 필요한 이유

기본 `StateBackend` 에이전트는 **대화 스레드가 끝나면 모든 정보를 잊습니다**.  
하지만 실제 어시스턴트는 아래 정보를 **대화 간에 유지**해야 합니다:

- 사용자 선호도 (코딩 스타일, 사용 언어)
- 프로젝트 컨벤션 (아키텍처 결정, 네이밍 규칙)
- 이전 대화에서 학습한 피드백
- 자주 참조하는 정보 (API 문서, 설정값)

### 해결 방식: CompositeBackend

![](../assets/images/composite_backend.png)

`/memories/` 경로에 저장된 파일은 **어떤 대화 스레드에서든** 접근할 수 있습니다.  
이전 노트북(00-backends)에서 배운 `CompositeBackend`를 활용하여 이를 구현합니다.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend, FilesystemBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver


# 1. 스토어와 체크포인터 생성, 실제 애플리케이션에서는 PostgresStore와 파일 또는 클라우드 기반 체크포인터를 사용할 수 있습니다.
store = InMemoryStore()          # 개발용 인메모리 스토어 (프로덕션: PostgresStore)
checkpointer = MemorySaver()     # 에이전트 상태 체크포인트 관리


# 2. CompositeBackend 팩토리 — /memories/만 영속, 나머지는 에페메럴(스레드 내 메모리, 스레드가 바뀌면 초기화 됨)
def memory_backend_factory(runtime):
    """경로 기반 백엔드 라우팅.
    
    /memories/* → StoreBackend (크로스 스레드 영속)
    그 외 경로 → StateBackend (스레드 내 에페메럴)
    """
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/memories/": StoreBackend(runtime),
        },
    )


# 3. 장기 메모리를 지원하는 에이전트 생성
memory_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 개인 어시스턴트입니다.
사용자가 기억해 달라고 하는 정보는 /memories/ 폴더에 저장하세요.
이전에 저장한 메모리가 있으면 참고하여 응답하세요.
한국어로 응답하세요.""",
    backend=memory_backend_factory,
    store=store,
    checkpointer=checkpointer,
)

print("장기 메모리 에이전트 생성 완료")

장기 메모리 에이전트 생성 완료


---
## 2. 크로스 스레드 메모리 공유

`StoreBackend`에 저장된 데이터는 **스레드 간에 공유**됩니다.  
아래 예제에서 스레드 1에서 저장한 선호도를 스레드 2에서 읽어봅니다.

이것이 장기 메모리의 핵심입니다: 새로운 대화를 시작해도 이전에 저장한 정보를 기억합니다.

In [11]:
# 스레드 1: 사용자 선호도를 /memories/ 경로에 저장합니다.
# /memories/ 경로는 CompositeBackend가 StoreBackend로 라우팅하므로 영속 저장됩니다.
config_t1 = {"configurable": {"thread_id": "memory-thread-1"}}

result1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": """
다음 정보를 기억해 주세요:
- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터, 타입 힌트 필수
/memories/preferences.txt에 저장해 주세요.
"""}]},
    config={**config_t1, **lf_config},
)

print("[스레드 1] 저장 결과:")
print(result1["messages"][-1].content)

[스레드 1] 저장 결과:
저장했습니다. 앞으로 이 선호를 참고하겠습니다.


In [12]:
# 스레드 2: 완전히 새로운 대화에서 이전에 저장한 메모리를 읽습니다.
# thread_id가 다르지만 StoreBackend 덕분에 동일 파일에 접근 가능합니다.
config_t2 = {"configurable": {"thread_id": "memory-thread-2"}}

result2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "/memories/preferences.txt를 읽어서 내 선호도를 알려 주세요."}]},
    config={**config_t2, **lf_config},
)

print("[스레드 2] 메모리 읽기 결과:")
print(result2["messages"][-1].content)

[스레드 2] 메모리 읽기 결과:
기억된 선호도는 다음과 같습니다.

- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터 사용, 타입 힌트 필수

원하시면 이 선호도를 바탕으로 앞으로 코드 예시나 답변 스타일도 맞춰드릴게요.


---
## 3. AGENTS.md를 통한 컨텍스트 주입

`memory` 파라미터를 사용하면, 에이전트가 시작될 때 **AGENTS.md 파일을 자동으로 로드**하여  
시스템 프롬프트에 주입합니다.

### AGENTS.md란?
에이전트에게 항상 적용되어야 하는 **규칙, 컨벤션, 컨텍스트 정보**를 담는 마크다운 파일입니다.  
Claude Code의 `CLAUDE.md`와 유사한 개념입니다.

### 특징
- 에이전트가 시작할 때 **항상 로드** (on-demand가 아님)
- `<agent_memory>` 태그로 시스템 프롬프트에 주입
- 여러 소스 지정 가능 (배열)
- 에이전트가 `edit_file` 도구로 AGENTS.md를 **스스로 업데이트** 가능

### 예시: AGENTS.md 내용
```markdown
# 프로젝트 컨벤션
- 코드 스타일: Black, isort, mypy strict
- 테스트: pytest, 커버리지 80% 이상
- 커밋 메시지: Conventional Commits 형식
```

In [13]:
import tempfile

# 임시 디렉토리 생성 — FilesystemBackend의 root_dir로 사용합니다.
# 실제 환경에서는 프로젝트 루트 디렉토리를 사용합니다.
tmp_dir = tempfile.mkdtemp()
print(f"임시 디렉토리 생성: {tmp_dir}")

임시 디렉토리 생성: /tmp/tmpu5srjlwn


In [19]:
# memory 파라미터: 에이전트 시작 시 자동으로 로드할 AGENTS.md 경로를 지정합니다.
# 이 파일의 내용이 시스템 프롬프트에 <agent_memory> 태그로 주입됩니다.
memory_inject_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 코딩 어시스턴트입니다.",
    #backend=FilesystemBackend(root_dir=tmp_dir, virtual_mode=True),
    # 눈으로 확인하기 위해 실제 프로젝트 루트를 사용합니다. 실제로는 tmp_dir을 사용하여 격리된 환경에서 테스트하는 것이 좋습니다.
    # virtual_mode=True로 설정하면 에이전트가 파일 시스템을 직접 수정하지 않고, 메모리 상에서만 파일을 읽고 쓸 수 있습니다. 실제 프로젝트 루트를 사용할 때는 virtual_mode=False로 설정하여 에이전트가 AGENTS.md를 실제로 읽고 쓸 수 있도록 해야 합니다.
    backend=FilesystemBackend(root_dir='./', virtual_mode=False),
    memory=["/memory/AGENTS.md"],  # AGENTS.md 파일 경로 (배열로 여러 파일 지정 가능)
)

# 에이전트가 AGENTS.md의 규칙을 따르는지 확인합니다.
# AGENTS.md에 정의된 프로젝트 컨벤션이 코드 생성에 반영되어야 합니다.
result = memory_inject_agent.invoke(
    #{"messages": [{"role": "user", "content": "간단한 HTTP 클라이언트 클래스를 만들어 주세요. 프로젝트 컨벤션을 따라주세요."}]},
    {"messages": [{"role": "user", "content": "AGENT.md 내 규칙을 따르는 코드를 생성해 주세요."}]},
    config=lf_config,
)

print(result["messages"][-1].content)

가능합니다. 다만 지금은 `AGENT.md`의 내용만 확인됐고, 어떤 코드를 생성해야 하는지 구체적인 요구사항이 없습니다.

`AGENT.md` 요약:
- 이 디렉토리는 실습용 스킬 폴더
- LangChain Agent가 사용할 수 있는 도구들 중 SKILL 소개
- 5일 교육 과정 중 4일차 실습
- 특정 실습 질문 시 앞뒤 실습 문맥까지 고려
- 현재 디렉토리: `/home/kmkang/posco-dx-agent-dev-day3/02-skills`

원하시는 것을 알려주세요:
- 생성할 코드의 언어
- 구현할 기능/과제
- 대상 파일 경로
- 기존 파일 수정인지, 새 파일 생성인지

예:
- “`memory` 폴더에 Python 예제 코드 작성”
- “AGENT.md 규칙에 맞춰 `README.md` 예시 추가”
- “현재 실습 주제에 맞는 LangChain 샘플 코드 생성”

원하시는 작업을 주시면 `AGENT.md` 규칙에 맞춰 바로 작성하겠습니다.


---
## 4. 스킬 (Skills)

스킬은 에이전트에게 **도메인 전문 지식**을 부여하는 모듈화된 지침 세트입니다.

### Memory vs Skills

| 비교 | Memory (AGENTS.md) | Skills (SKILL.md) |
|------|-------------------|-------------------|
| **로딩** | 항상 로드 (Always) | 필요 시 로드 (On-demand) |
| **파일 형식** | `AGENTS.md` | `SKILL.md` (YAML 프론트매터) |
| **적합한 용도** | 항상 적용되는 규칙/컨벤션 | 특정 태스크에 필요한 큰 컨텍스트 |
| **토큰 효율** | 항상 소비 | 점진적 공개로 절약 |
| **크기** | 간결할수록 좋음 | 대용량 가능 (10MB 제한) |
| **업데이트** | 에이전트가 edit_file로 수정 가능 | 보통 정적 |

### Progressive Disclosure (점진적 공개)

스킬은 한 번에 전부 로드하지 않습니다:
1. **1단계**: 프론트매터(이름, 설명)만 로드 → 토큰 소비 최소화
2. **2단계**: 사용자 요청과 관련된 스킬을 **에이전트가 판단**
3. **3단계**: 필요한 스킬의 **전체 내용**을 그때 로드

> 💡 이 방식으로 수십 개의 스킬을 등록해도 토큰을 절약하면서 필요한 전문 지식에 접근할 수 있습니다.

### SKILL.md 구조

```yaml
---
name: web-research           # 스킬 식별자 (최대 64자, 소문자+하이픈)
description: >               # 설명 (최대 1024자) — 매칭에 사용
  체계적인 웹 리서치를 수행하기 위한 단계별 가이드.
  정보 수집, 검증, 정리까지의 전체 워크플로를 다룹니다.
license: MIT
compatibility: Python 3.8+
metadata:
  category: research
allowed-tools: ls read_file write_file
---

# Web Research 스킬

## 사용 시기
- 사용자가 특정 주제에 대한 조사를 요청할 때
- 최신 정보가 필요한 질문이 들어올 때

## 워크플로
1. 검색 쿼리 설계
2. 다양한 소스에서 정보 수집
3. 정보 교차 검증
4. 구조화된 보고서 작성
```

In [39]:
# skills 파라미터: 에이전트가 접근할 수 있는 스킬 디렉토리를 지정합니다.
# 디렉토리 내의 SKILL.md 파일들이 자동으로 발견됩니다.
# Progressive Disclosure: 처음에는 프론트매터만 로드하고, 필요 시 전체 내용을 로드합니다.
skilled_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 시니어 개발자입니다. 사용 가능한 스킬을 활용하여 작업을 수행하세요.",
    #backend=FilesystemBackend(root_dir=tmp_dir, virtual_mode=True),
    backend=FilesystemBackend(root_dir='./', virtual_mode=True),
    skills=["/skills/skills"],  # 스킬 소스 디렉토리 (배열로 여러 경로 지정 가능)
)

print("스킬 에이전트 생성 완료")

스킬 에이전트 생성 완료


In [41]:
# 스킬을 활용한 코드 리뷰 요청
# 에이전트가 관련 스킬(예: code-review 스킬)을 자동으로 판단하여 로드합니다.
# SQL Injection 등 보안 취약점을 포함한 코드를 제공하여 리뷰 품질을 확인합니다.
result = skilled_agent.invoke(
    #{"messages": [{"role": "user", "content": "사용할 수 있는 /skills/skills 폴더 하위의 skills 목록 보여줘."}]},
    {"messages": [{"role": "user", "content": "프론트엔드 디자인에 대해 사용할 수 있는 스킬 목록을 기반으로 실무에 적용할 방안을 설명해줘."}]},
    config=lf_config,
)

print(result["messages"][-1].content)

프론트엔드 디자인에 직접적으로 적용 가능한 스킬은 주로 아래입니다.

- **frontend-design**
- **web-artifacts-builder**
- **theme-factory**
- **brand-guidelines**
- **webapp-testing**
- **canvas-design**
- 경우에 따라 **algorithmic-art** / **slack-gif-creator** / **pptx** / **docx** 등도 보조적으로 활용 가능

실무 적용 방안은 이렇게 보면 됩니다.

## 1) frontend-design
가장 핵심입니다.  
**용도:** 실제 제품 화면, 랜딩 페이지, 대시보드, 컴포넌트 UI를 고품질로 설계하고 구현할 때 사용합니다.

**실무 적용**
- 신규 서비스 랜딩 페이지 제작
- 관리자/대시보드 UI 리디자인
- React 기반 컴포넌트 구조 설계
- “그럴듯한 샘플”이 아니라 **배포 가능한 수준의 UI** 초안 생성
- 디자인 시스템에 맞는 레이아웃/타이포그래피/간격/컬러 적용

**권장 사용 시점**
- PM/디자이너가 와이어프레임만 준 상태
- 개발 초기에 화면 구조를 빠르게 만들어야 할 때
- 기존 UI가 너무 평범해서 차별화가 필요할 때

---

## 2) web-artifacts-builder
복잡한 웹 아티팩트가 필요할 때 유용합니다.  
**용도:** 상태 관리, 인터랙션, 다중 컴포넌트, 페이지 전환이 있는 고도화된 HTML/React 산출물 제작.

**실무 적용**
- 인터랙티브 프로토타입
- 복잡한 제품 데모
- 여러 섹션이 있는 마케팅 페이지
- 내부 검토용 시뮬레이션 툴
- 단순 정적 HTML보다 더 풍부한 상호작용이 필요할 때

**권장 사용 시점**
- “이 화면이 실제로 어떻게 동작하는지” 보여줘야 할 때
- 기획/개발/디자인 합의가 필요할 때
- 단일 컴포넌트 수준을 넘어선 데모가 필요할 때

---

## 3) theme-factory
브랜드 톤을 빠르게 입힐 때 좋

---
## 5. 스킬 소스 우선순위

여러 스킬 소스를 지정하면, **나중에 나온 소스가 우선**합니다 (last wins).

```python
skills=[
    "/skills/base/",     # 기본 스킬 (최저 우선순위)
    "/skills/user/",     # 사용자 스킬 (base 덮어쓰기 가능)
    "/skills/project/",  # 프로젝트 스킬 (최우선)
]
```

같은 이름의 스킬이 여러 소스에 있으면, 마지막 소스의 스킬이 사용됩니다.

> 💡 이 구조를 활용하면 조직 공통 스킬(base) 위에 팀/프로젝트별 커스터마이징을 레이어링할 수 있습니다.

---
## 6. 서브에이전트의 스킬 상속

메인 에이전트가 서브에이전트를 생성할 때 스킬 상속 방식이 다릅니다.

| 서브에이전트 유형 | 스킬 상속 |
|-------------------|----------|
| General-purpose (빌트인) | 메인 에이전트의 스킬을 **자동 상속** |
| 커스텀 SubAgent | **명시적 `skills` 파라미터** 필요 |

```python
# 커스텀 서브에이전트에 스킬을 명시적으로 부여하는 예시
subagent = {
    "name": "reviewer",
    "description": "코드 리뷰 전문 에이전트",
    "system_prompt": "...",
    "tools": [],
    "skills": ["/skills/code-review/"],  # 필요한 스킬만 명시적으로 지정
}
```

> 빌트인 서브에이전트는 자동 상속이 편리하지만, 커스텀 서브에이전트는 필요한 스킬만 선택적으로 부여하여 토큰을 절약할 수 있습니다.

---
## 핵심 정리

| 항목 | 내용 |
|------|------|
| 장기 메모리 | `CompositeBackend` + `StoreBackend`로 `/memories/` 영속화 |
| AGENTS.md | `memory=["/path/AGENTS.md"]` → 항상 시스템 프롬프트에 주입 |
| Skills | `skills=["/skills/"]` → SKILL.md 기반 점진적 공개 |
| Progressive Disclosure | 프론트매터만 먼저 로드 → 필요 시 전체 로드 (토큰 절약) |
| 스킬 우선순위 | 나중 소스가 우선 (last wins) |
| Memory vs Skills | Memory = 항상 로드, 간결 / Skills = 필요 시 로드, 대용량 가능 |

### 실무 가이드
- **항상 적용되는 규칙** (코딩 컨벤션, 프로젝트 정책) → AGENTS.md
- **특정 작업에만 필요한 전문 지식** (코드 리뷰, 리서치 프로토콜) → SKILL.md
- **사용자 개인 정보** (선호도, 히스토리) → CompositeBackend + /memories/